In [5]:
# These lines are to turn individual pdbs or cifs into FASTA files for DB searching etc.

from pathlib import Path
from Bio import SeqIO

from pathlib import Path
from Bio import SeqIO

def structure_to_fasta(in_path: str | Path, out_path: str | Path | None = None):
    in_path = Path(in_path)
    if out_path is None:
        out_path = in_path.with_suffix(".fasta")

    # For prediction files, use ATOM-based parsing
    records = list(SeqIO.parse(str(in_path), "pdb-atom"))
    if not records:
        raise ValueError(f"No ATOM records found in {in_path}")

    SeqIO.write(records, str(out_path), "fasta")
    print(f"Extracted {len(records)} chains from {in_path.name}")
    return out_path


# ---------------------
# Example usage:
# ---------------------
structure_to_fasta("/Users/emma/Desktop/predictions-RFD3/E8-preds/converted_pdbs/fixed-oxygens-E8-separated-chains_LA_HOH_binding_site_0_model_7.pdb")
structure_to_fasta("/Users/emma/Desktop/predictions-RFD3/E8-preds/converted_pdbs/fixed-oxygens-E8-separated-chains_LA_HOH_binding_site_0_model_2.pdb")


Extracted 1 chains from fixed-oxygens-E8-separated-chains_LA_HOH_binding_site_0_model_7.pdb
Extracted 1 chains from fixed-oxygens-E8-separated-chains_LA_HOH_binding_site_0_model_2.pdb


/Users/emma/Desktop/protein-design-lanthanides/.venvpy3-11/lib/python3.11/site-packages/Bio/PDB/Atom.py:237: PDBConstructionWarning: Used element 'LA' for Atom (name=LA) with given element 'A'
  warnings.warn(msg, PDBConstructionWarning)
/Users/emma/Desktop/protein-design-lanthanides/.venvpy3-11/lib/python3.11/site-packages/Bio/SeqIO/PdbIO.py:337: BiopythonParserWarning: 'HEADER' line not found; can't determine PDB ID.
  warnings.warn(
/Users/emma/Desktop/protein-design-lanthanides/.venvpy3-11/lib/python3.11/site-packages/Bio/PDB/Atom.py:237: PDBConstructionWarning: Used element 'LA' for Atom (name=LA) with given element 'A'
  warnings.warn(msg, PDBConstructionWarning)
/Users/emma/Desktop/protein-design-lanthanides/.venvpy3-11/lib/python3.11/site-packages/Bio/SeqIO/PdbIO.py:337: BiopythonParserWarning: 'HEADER' line not found; can't determine PDB ID.
  warnings.warn(


PosixPath('/Users/emma/Desktop/predictions-RFD3/E8-preds/converted_pdbs/fixed-oxygens-E8-separated-chains_LA_HOH_binding_site_0_model_2.fasta')

In [ ]:
# This lines are for the same purpose but in batch
def batch_structures_to_fasta(input_dir: str | Path,
                              output_dir: str | Path,
                              single_output: bool = False):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if single_output:
        merged_path = output_dir / "all_sequences.fasta"
        if merged_path.exists():
            merged_path.unlink()

    total_chains = 0
    for path in input_dir.iterdir():
        if path.suffix.lower() != ".pdb":  # Adjust if you have CIFs
            continue

        records = list(SeqIO.parse(str(path), "pdb-atom"))
        if not records:
            print(f"Skipped {path.name}: no ATOM records")
            continue

        total_chains += len(records)
        if single_output:
            for rec in records:
                rec.id = f"{path.stem}|{rec.id}"
                rec.description = ""
            with open(merged_path, "a") as handle:
                SeqIO.write(records, handle, "fasta")
        else:
            out_path = output_dir / f"{path.stem}.fasta"
            SeqIO.write(records, str(out_path), "fasta")

    print(f"Processed {total_chains} total chains")


# ---------------------------
# Example usage:
# ---------------------------

batch_structures_to_fasta("PDBs", "FASTAs", single_output=False)
# or:
batch_structures_to_fasta("PDBs", "FASTAs", single_output=True)
